# WIA3003: Academic Project II
## Effect of Recorded Ethnicity on Sex Estimation

Research question: does recorded ethnicity affect sex estimation from the eight orbital and nasal measurements? This responds to the AP1 panel comments. One panel asked for a comparison by ethnicity. The other stated that sex estimation is not population specific.

Three checks use logistic regression, the strongest baseline so far, with repeated grouped cross-validation:

- A. Does adding ethnicity as an input change performance?
- B. Does a model trained on two groups work on the third?
- C. How much does performance differ between groups, with uncertainty?

Notebook 11 described one test split. This notebook uses every record as test data several times. Claude assisted with writing and running this notebook.

## 1.0 Load Dataset

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

folder = Path.cwd()
project_folder = folder.parent if folder.name == "notebooks" else folder
details = json.loads((project_folder / "results" / "qcnn_baseline" / "run_details.json").read_text(encoding="utf-8"))

data_path = Path.home() / "Downloads" / "Orbital, Nasal and Craniofacial Class Pattern.xls"
dataset_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if dataset_hash != details["dataset_sha256"]:
    raise ValueError("Workbook changed since the grouped QCNN run. Verify the data first.")

df = pd.read_excel(data_path, sheet_name="Raw Data")
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
features = details["features"]
X = df[features].apply(pd.to_numeric, errors="raise")
X["Ethnicity"] = df["Ethnicity"].astype("string").str.strip().str.title().astype(str)
y = df["Gender"].astype("string").str.strip().map(details["label_mapping"]).astype(int)
groups = X.groupby(features, sort=False, dropna=False).ngroup().to_numpy()
ethnicities = sorted(X["Ethnicity"].unique())

display(pd.crosstab(X["Ethnicity"], y.map({0: "Female", 1: "Male"}), margins=True, margins_name="Total"))

Gender,Female,Male,Total
Ethnicity,,,
Chinese,176,204,380
Indian,100,182,282
Malay,179,201,380
Total,455,587,1042


## 2.0 Settings

Use five-fold grouped cross-validation, repeated with five seeds. Folds are stratified by sex and ethnicity together, so each fold keeps both proportions. Records with identical measurements stay in the same fold. Settings are fixed before running.

In [2]:
SEEDS = [42, 123, 2026, 7, 99]
FOLDS = 5
strata = y.astype(str) + "_" + X["Ethnicity"]

def logistic():
    return LogisticRegression(solver="lbfgs", C=1.0, max_iter=2000)

def add_interactions(matrix):
    # Columns: eight scaled measurements, then the ethnicity indicators.
    measurements, indicators = matrix[:, :len(features)], matrix[:, len(features):]
    products = (measurements[:, :, None] * indicators[:, None, :]).reshape(len(matrix), -1)
    return np.hstack([matrix, products])

def make_model(variant):
    scaled = ("scale", StandardScaler(), features)
    if variant == "Measurements only":
        return Pipeline([("columns", ColumnTransformer([scaled])), ("logistic", logistic())])
    encoded = ("ethnicity", OneHotEncoder(drop="first", sparse_output=False), ["Ethnicity"])
    steps = [("columns", ColumnTransformer([scaled, encoded]))]
    if variant == "Measurements + ethnicity + interactions":
        steps.append(("interactions", FunctionTransformer(add_interactions)))
    return Pipeline(steps + [("logistic", logistic())])

VARIANTS = ["Measurements only", "Measurements + ethnicity", "Measurements + ethnicity + interactions"]

def folds(seed):
    splitter = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    for fold, (train_rows, test_rows) in enumerate(splitter.split(X, strata, groups)):
        assert not set(groups[train_rows]) & set(groups[test_rows])
        yield fold, train_rows, test_rows

## 3.0 Part A: Ethnicity as an Input

Compare three models on the same folds. Adding ethnicity lets the model shift its decision threshold per group. Adding interactions lets each measurement carry a different weight per group, which is close to fitting a separate model for each group.

In [3]:
fold_rows = []
predictions = {}
for seed in SEEDS:
    for fold, train_rows, test_rows in folds(seed):
        for variant in VARIANTS:
            model = make_model(variant).fit(X.iloc[train_rows], y.iloc[train_rows])
            predicted = model.predict(X.iloc[test_rows])
            fold_rows.append({
                "Seed": seed, "Fold": fold, "Model": variant,
                "Accuracy": accuracy_score(y.iloc[test_rows], predicted),
                "Macro F1": f1_score(y.iloc[test_rows], predicted, average="macro"),
            })
            predictions.setdefault((variant, seed), np.empty(len(X), dtype=int))[test_rows] = predicted
fold_scores = pd.DataFrame(fold_rows)

input_summary = fold_scores.groupby("Model")[["Accuracy", "Macro F1"]].agg(["mean", "std"]).reindex(VARIANTS)
display(input_summary.round(4))

paired = fold_scores.pivot(index=["Seed", "Fold"], columns="Model", values="Macro F1")
input_difference = pd.DataFrame([{
    "Comparison": f"{variant} minus measurements only",
    "Mean macro F1 difference": (paired[variant] - paired[VARIANTS[0]]).mean(),
    "Std of difference": (paired[variant] - paired[VARIANTS[0]]).std(),
    "Folds improved": int((paired[variant] > paired[VARIANTS[0]]).sum()),
    "Folds worse": int((paired[variant] < paired[VARIANTS[0]]).sum()),
    "Folds": len(paired),
} for variant in VARIANTS[1:]]).set_index("Comparison")
display(input_difference.round(4))

Accuracy         Macro F1        
                                            mean     std     mean     std
Model                                                                    
Measurements only                         0.7649  0.0263   0.7600  0.0271
Measurements + ethnicity                  0.7651  0.0239   0.7603  0.0247
Measurements + ethnicity + interactions   0.7725  0.0260   0.7680  0.0261

,Mean macro F1 difference,Std of difference,Folds improved,Folds worse,Folds
Comparison,,,,,
Measurements + ethnicity minus measurements only,0.0003,0.0139,11,10,25
Measurements + ethnicity + interactions minus measurements only,0.0080,0.0159,18,6,25


## 4.0 Part C: Performance by Ethnicity

Use the measurements-only model. Each repeat gives one out-of-fold prediction for every record. Score each ethnicity group in each repeat, then report the mean and standard deviation across the five repeats.

In [4]:
def group_scores(labels, predicted):
    return {
        "Accuracy": accuracy_score(labels, predicted),
        "Macro F1": f1_score(labels, predicted, average="macro"),
        "Female recall": recall_score(labels, predicted, pos_label=0),
        "Male recall": recall_score(labels, predicted, pos_label=1),
    }

group_rows = []
for seed in SEEDS:
    predicted = predictions[(VARIANTS[0], seed)]
    for ethnicity in ethnicities:
        rows = (X["Ethnicity"] == ethnicity).to_numpy()
        group_rows.append({"Seed": seed, "Ethnicity": ethnicity, "Records": int(rows.sum()),
                           **group_scores(y[rows], predicted[rows])})
group_runs = pd.DataFrame(group_rows)
within_group = group_runs.groupby("Ethnicity").agg(
    Records=("Records", "first"),
    **{f"{metric} {stat}": (metric, stat)
       for metric in ("Accuracy", "Macro F1", "Female recall", "Male recall") for stat in ("mean", "std")})
display(within_group.round(4))

,Records,Accuracy mean,Accuracy std,Macro F1 mean,Macro F1 std,Female recall mean,Female recall std,Male recall mean,Male recall std
Ethnicity,,,,,,,,,
Chinese,380,0.7616,0.0051,0.7586,0.0051,0.7023,0.0051,0.8127,0.0064
Indian,282,0.7879,0.0077,0.7731,0.0086,0.7500,0.0158,0.8088,0.0046
Malay,380,0.7511,0.0030,0.7496,0.0030,0.7151,0.0056,0.7831,0.0044


## 5.0 Part B: Leave One Ethnicity Out

Train on two groups and test on all records of the third. Compare with the Part C score for the same group, where the model had also seen records from that group. A clear drop means the model does not transfer between groups.

In [5]:
transfer_rows = []
for held_out in ethnicities:
    test_rows = (X["Ethnicity"] == held_out).to_numpy()
    model = make_model(VARIANTS[0]).fit(X[~test_rows], y[~test_rows])
    scores = group_scores(y[test_rows], model.predict(X[test_rows]))
    transfer_rows.append({
        "Held-out ethnicity": held_out,
        "Trained on": " + ".join(name for name in ethnicities if name != held_out),
        "Training records": int((~test_rows).sum()), "Test records": int(test_rows.sum()),
        **{f"Unseen group {metric.lower()}": value for metric, value in scores.items()},
        "Seen group macro F1": within_group.loc[held_out, "Macro F1 mean"],
        "Seen group macro F1 std": within_group.loc[held_out, "Macro F1 std"],
    })
transfer = pd.DataFrame(transfer_rows).set_index("Held-out ethnicity")
transfer["Macro F1 change when unseen"] = transfer["Unseen group macro f1"] - transfer["Seen group macro F1"]
display(transfer.round(4))

,Trained on,Training records,Test records,Unseen group accuracy,Unseen group macro f1,Unseen group female recall,Unseen group male recall,Seen group macro F1,Seen group macro F1 std,Macro F1 change when unseen
Held-out ethnicity,,,,,,,,,,
Chinese,Indian + Malay,662,380,0.7684,0.7658,0.7159,0.8137,0.7586,0.0051,0.0072
Indian,Chinese + Malay,760,282,0.7518,0.7408,0.7700,0.7418,0.7731,0.0086,-0.0322
Malay,Chinese + Indian,662,380,0.7553,0.7527,0.6927,0.8109,0.7496,0.0030,0.0031


## 6.0 Save Results

In [6]:
output_folder = project_folder / "results" / "ethnicity_effect"
output_folder.mkdir(parents=True, exist_ok=True)
fold_scores.to_csv(output_folder / "input_fold_scores.csv", index=False)
input_summary.to_csv(output_folder / "input_summary.csv")
input_difference.to_csv(output_folder / "input_difference.csv")
group_runs.to_csv(output_folder / "group_runs.csv", index=False)
within_group.to_csv(output_folder / "within_group_summary.csv")
transfer.to_csv(output_folder / "leave_one_ethnicity_out.csv")
settings = {
    "dataset_sha256": dataset_hash, "features": features, "seeds": SEEDS, "folds": FOLDS,
    "method": "StratifiedGroupKFold, strata sex x ethnicity, groups of identical measurements",
    "model": "StandardScaler + LogisticRegression(C=1.0)", "sklearn_version": sklearn.__version__,
}
(output_folder / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print("Saved results to:", output_folder)

Saved results to: C:\Users\jwenn\Desktop\UM\FYP\FYP-Project\results\ethnicity_effect
